# NHOS WS Demo

這份 Notebook 示範如何連到 New Horizons Desktop Backend 的 WebUI 串流端點，登入後訂閱指定 `device_uid`，並持續接收 `snapshot` 與 `update`。

對應的後端端點是 `GET/WS /newhorizons/stream/ws`，授權可用 `Authorization: Bearer <token>` 或 `?token=<token>`。

## 先決條件

- Desktop Backend 已啟動，預設是 `http://127.0.0.1:5051`
- 你知道可用的 `username` / `password`
- 你要訂閱的 `device_uid`
- 已安裝 `requests` 與 `websocket-client`

如果環境裡剛好有舊的 `socketio` 套件，不要用它。這個後端是原生 WebSocket，不是 Socket.IO。

In [ ]:
# 如未安裝，先執行：
# %pip install requests websocket-client pandas

import json
from datetime import datetime

import requests
import websocket


In [ ]:
BASE_URL = "http://127.0.0.1:5051"
USERNAME = "admin"
PASSWORD = "admin"
DEVICE_UID = "3CDC7545CCD0"

WS_URL = BASE_URL.replace("http://", "ws://").replace("https://", "wss://") + "/newhorizons/stream/ws"
WS_URL

In [ ]:
resp = requests.post(
    f"{BASE_URL}/api/token",
    json={"username": USERNAME, "password": PASSWORD},
)
resp.raise_for_status()

token_payload = resp.json()
token = token_payload["token"]
expires_in = token_payload.get("expires_in")

print("token 取得成功")
print(f"expires_in: {expires_in}")
print(f"token[:40]: {token[:40]}...")

## 建立串流連線

後端在連線後會先送出 `hello_ack`，接著用 `subscribe` 訂閱指定 `device_uid`。成功後通常會收到 `subscribed`，如果該設備已有最新資料，還會接到 `snapshot`。

In [ ]:
ws = websocket.create_connection(
    WS_URL,
    header=[f"Authorization: Bearer {token}"],
    timeout=10,
)
ws.settimeout(1.0)

hello_ack = json.loads(ws.recv())
hello_ack

In [ ]:
ws.send(json.dumps({"type": "subscribe", "device_uid": DEVICE_UID}, separators=(",", ":")))

subscribed = json.loads(ws.recv())
subscribed

## 持續接收資料

下面這個 loop 會把每筆事件收進 `events`，並即時印出。事件格式跟後端實作一致，常見類型如下：

- `snapshot`：訂閱後的最新快照
- `update`：後續增量更新
- `stream_ended`：設備不再可用或取消訂閱
- `error`：訊息格式或權限錯誤

In [ ]:
events = []

try:
    while True:
        raw = ws.recv()
        event = json.loads(raw)
        events.append({"received_at": datetime.now().isoformat(timespec="seconds"), **event})
        print(event)
except KeyboardInterrupt:
    print("手動停止串流")
except websocket.WebSocketTimeoutException:
    print("等待逾時，沒有新事件")
finally:
    ws.close()
    print("WebSocket 已關閉")

## 轉成資料表

如果你想做後續分析，可以把 `events` 轉成 `pandas.DataFrame`。對 `snapshot` 與 `update` 這類事件，常見資料會在 `data` 欄位中。

In [ ]:
try:
    import pandas as pd

    df = pd.DataFrame(events)
    df.head()
except ImportError:
    print("未安裝 pandas，略過 DataFrame 轉換")

## 補充說明

如果你要接遠端環境，把 `BASE_URL` 改成實際位址即可；若是 HTTPS，Notebook 會自動轉成 `wss://`。如果訂閱不到資料，先確認 `DEVICE_UID` 是否正確，以及該設備在後端是否已經有可用的 visualization snapshot。